In [ ]:
import re
import sys
import os


shared_schema = "00_shared"
try:
    catalog
except NameError:
    catalog = "vectorcatalog02"

spark.sql(f"USE CATALOG {catalog}")

# Get the current user's email from Spark
username = spark.sql("SELECT current_user()").first()[0]

# Strip off anything after '@'
safe_username = username.split('@')[0]

# Sanitize for use as a schema name (replace non-alphanumeric chars with _)
safe_username = re.sub(r'[^a-z0-9]', '_', safe_username.lower()).strip('_')

try:
    schema
except NameError:
    schema = safe_username

# Schema won't exist yet during participant setup
try:
    spark.sql(f"USE SCHEMA {schema}")
except:
    print(f"Schema {schema} doesn't exist yet.  To be expected when running 00_setup_participant notebook")

shared_volume = f"/Volumes/{catalog}/{shared_schema}/raw_files"
wheel_volume = f"/Volumes/{catalog}/{shared_schema}/wheels"

# Genie Space ID — set by instructor after running Step 9 of 00_instructor_setup.ipynb
genie_space_id = "FILL_IN_AFTER_INSTRUCTOR_SETUP"

print("="*50)
print(f"Your Databricks login (username)      : {username}")
print(f"Your safe username (safe_username)    : {safe_username}")
print(f"Your schema name (catalog).(schema)   : {catalog}.{schema}")
print("\n")
print(f"Shared schema name (catalog).(shared_schema) : {catalog}.{shared_schema}")
print(f"Shared volume path (shared_volume)           : {shared_volume}")
print('=' * 50)
print("\n")

In [0]:
def replicate_artifacts(force = False):

    import shutil, os, time

    # check to see if the user's home directory already has the artifacts
    if os.path.exists(f"/Workspace/Users/{username}/Production-ready-ML-workshop"):
        if force:
            print("Removing existing artifacts from your home directory.")
            shutil.rmtree(f"/Workspace/Users/{username}/Production-ready-ML-workshop")
            # Allow WSFS async flush to complete before writing new files
            time.sleep(5)
        else:
            print("Artifacts already exist in your home directory.  Skipping copy.")
            return

    # Copy the entire participant file tree into the user's home directory
    artifact_src = f"./Production-ready-ML-workshop/participant"

    shutil.copytree(
        f"{artifact_src}",
        f"/Workspace/Users/{username}/Production-ready-ML-workshop",
        dirs_exist_ok=True
    )
    print("Copied artifacts to your home directory.")
